In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import os
from pathlib import Path
import re
import sys
import tempfile
import time

import finufft
from IPython.display import HTML
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
try:
    import pyvista as pv
except ImportError:
    pv = None
from numpy.fft import fftn, fftshift, ifftn, ifftshift, fftfreq, fftshift
from scipy.ndimage import uniform_filter
from scipy import ndimage as ndi
from scipy.stats import norm, ncx2, chi2, multivariate_normal
from tqdm.auto import tqdm
import visu3d as v3d
import zarr

from sptflux import flux_probing, ihpp_data, plotting, ihpp_sim, datastructs, utils, logger, config
import plotly.io as pio

pio.renderers.default = "browser"          # opens in your default browser tab
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "Times New Roman",
    # "font.serif": ["Computer Modern Roman"], # The default LaTeX font
})

logger = logger.get_logger(__name__)

# normality plots

In [ ]:
import numpy as np
import finufft
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FixedLocator, MaxNLocator
from scipy.stats import norm

from sptflux import ihpp_data, ihpp_sim

In [ ]:
def flux_func_gaussian_ball_linear(
    T_exp,
    start_pos=(0.0, 1.0),   # (x, y) in normalised [0,1] coords, top-left
    end_pos=(1.0, 0.0),     # (x, y) in normalised [0,1] coords, bottom-right
    sigma=0.1,
    max_amplitude=1e4,
    normalize=False,
):
    def flux(x, y, t):
        # linear interpolation of centre position
        frac = t / T_exp if not normalize else t   # t is already in [0,1] if normalize
        curr_x = start_pos[0] + (end_pos[0] - start_pos[0]) * frac
        curr_y = start_pos[1] + (end_pos[1] - start_pos[1]) * frac
        return max_amplitude * np.exp(
            -((x - curr_x) ** 2 + (y - curr_y) ** 2) / (2 * sigma ** 2)
        )

    return flux, max_amplitude

T_exp = 1.0
GAUSS_LINEAR_BASE = dict(
    start_pos=(0.2, 0.2),
    end_pos=(0.8, 0.8),
    T_exp=T_exp,
    sigma=0.1,
    normalize=False
)

def mode_index(mode, shape):
    """
    Convert a signed-frequency mode tuple (ft, fy, fx) to
    the numpy array index produced by np.fft (output is
    arranged like np.fft.fftfreq, i.e. 0..N//2, -(N//2)...-1).
    NOT nufft
    """
    return tuple(int(m) % int(s) for m, s in zip(mode, shape))

def gaussian_contour_levels(mu, cov2, ax, n_sigma=(1, 2), color="cyan", lw=0.8):
    """Draw elliptical contours of a 2-D Gaussian on ax."""
    vals, vecs = np.linalg.eigh(cov2)
    vals = np.maximum(vals, 0.0)
    theta = np.linspace(0, 2 * np.pi, 300)
    max_sigma = max(n_sigma)
    base_rgb = np.array(mcolors.to_rgb(color))
    for ns in n_sigma:
        t = (ns - 1) / max_sigma                          # 0 at innermost, ~1 at outermost
        faded = base_rgb + (1 - base_rgb) * t * 0.7      # blend toward white
        ell = ns * np.stack([np.sqrt(vals[0]) * np.cos(theta),
                             np.sqrt(vals[1]) * np.sin(theta)])
        xy = vecs @ ell
        ax.plot(mu[0] + xy[0], mu[1] + xy[1],
                color=faded, lw=lw, linestyle="--")

## continuous space normality

In [ ]:
# ======================================================
# Configuration
# ======================================================

# Amplitude levels to sweep (scales max_amplitude)
# hardcode attempt to keep E[nphotons] = 5, 50, 500
AMPLITUDE_LEVELS = (5 / 6.2) * np.array([1e2, 1e3, 1e4, 1e5])

N_TRIALS = 10000          # number of Monte-Carlo trials per amplitude level

# Fourier modes to probe (t-freq, y-freq, x-freq) — zero-based, centred convention
PROBE_MODES = [
    (1, 0, 2),
    (1, 0, 1),
    (1, 1, 0),
    (1, 2, 0),
    (2, 2, 2),
    (3, 1, 1),
    (5, 3, 7),
]

# Base Gaussian-ball args (max_amplitude will be overridden in the loop)
# GAUSS_BASE = dict(
#     height=1,
#     width=1,
#     x_dist=0.4,
#     y_dist=0.4,
#     T_exp=T_exp,
#     hz_move_x=4.5,
#     hz_move_y=4.5,
#     hz_light=0,
#     sigma=0.1,
#     normalize=False,
# )
GAUSS_BASE = dict(
    height=1,
    width=1,
    x_dist=0.4,
    y_dist=0.4,
    T_exp=T_exp,
    hz_move_x=3.0,
    hz_move_y=0,
    hz_light=2.0,
    sigma=0.1,
    normalize=False,
)
GAUSS_LINEAR_BASE = dict(
    start_pos=(0.2, 0.2),
    end_pos=(0.8, 0.8),
    T_exp=T_exp,
    sigma=0.1,
    normalize=False
)

NUFFT_EPS = 1e-6

# Reconstruction grid sizes
n_modes = (12, 64, 16)  # we are not probing up to the resolution Nyquist
Nt, Ny, Nx = 10000, 100, 100
# tt, yy, xx = np.linspace(0, 1, Nt, endpoint=False), np.linspace(0, 1, Ny, endpoint=False), np.linspace(0, 1, Nx, endpoint=False)
# TT, YY, XX = np.meshgrid(tt, yy, xx, indexing="ij")

# fluxgt = ihpp_data.flux_func_gaussian_ball(**GAUSS_BASE, max_amplitude=1.0)[0](XX, YY, TT)
fluxfuncgt = flux_func_gaussian_ball_linear(**GAUSS_LINEAR_BASE, max_amplitude=1.0)[0]
# fluxgt = fluxfuncgt(XX, YY, TT)


# ======================================================
# Main simulation loop
# ======================================================

# Storage: coefficients[amp_idx][mode_idx] = list of complex values
coeffs = {
    ai: {mi: [] for mi in range(len(PROBE_MODES))}
    for ai in range(len(AMPLITUDE_LEVELS))
}
mean_photons = {ai: [] for ai in range(len(AMPLITUDE_LEVELS))}

for ai, amp in enumerate(AMPLITUDE_LEVELS):
    print(f"Amplitude {amp:.0e}  ({ai+1}/{len(AMPLITUDE_LEVELS)})")

    # lam, lmax = ihpp_data.flux_func_gaussian_ball(
    #     max_amplitude=amp, **GAUSS_BASE
    # )
    lam, lmax = flux_func_gaussian_ball_linear(
        max_amplitude=amp, **GAUSS_LINEAR_BASE
    )

    for trial in tqdm(range(N_TRIALS)):
        points = ihpp_sim.simulate_ihpp3d(
            lambda_fn=lam,
            s_region=[[0, 0], [1, 0], [1, 1], [0, 1]],
            t_range=(0, 1),
            npoints=None,
            nx=100, ny=100, nt=100,
            lmax=lmax,
            rng=None,          # random seed each trial
        )

        n_photons = len(points)
        mean_photons[ai].append(n_photons)

        if n_photons == 0:
            # No photons: all coefficients are zero
            for mi in range(len(PROBE_MODES)):
                coeffs[ai][mi].append(0.0 + 0.0j)
            continue

        px, py, pt = points[:, 0], points[:, 1], points[:, 2]

        # # Discretize points onto grid
        # px_d = np.round(px * (Nx - 1)).astype(int)
        # py_d = np.round(py * (Ny - 1)).astype(int)
        # pt_d = np.round(pt * (Nt - 1)).astype(int)

        # # Drop duplicates by converting to a set of tuples
        # unique_points = np.unique(np.stack([px_d, py_d, pt_d], axis=1), axis=0)
        # # print(f"Trial {trial}: {n_photons} photons, {len(unique_points)} unique grid points (percent lost: {(n_photons - len(unique_points)) / n_photons:.1%})")
        # n_photons_disc = len(unique_points)

        # px = unique_points[:, 0] / (Nx - 1)
        # py = unique_points[:, 1] / (Ny - 1)
        # pt = unique_points[:, 2] / (Nt - 1)

        F = finufft.nufft3d1(
            pt * 2 * np.pi,
            py * 2 * np.pi,
            px * 2 * np.pi,
            np.ones(n_photons, dtype="complex128"),
            n_modes=n_modes,
            eps=NUFFT_EPS,
            isign=-1,
        )

        for mi, mode in enumerate(PROBE_MODES):
            idx = mode_index(mode, F.shape)
            coeffs[ai][mi].append(np.fft.ifftshift(F)[idx])

    avg = np.mean(mean_photons[ai])
    print(f"  mean photons = {avg:.1f}")

# Convert to arrays
for ai in range(len(AMPLITUDE_LEVELS)):
    for mi in range(len(PROBE_MODES)):
        coeffs[ai][mi] = np.array(coeffs[ai][mi])

In [ ]:
k = 10
# Flat indices (into flattened array)
flat_indices = np.argpartition(np.fft.ifftshift(F).ravel(), -k)[-k:]
flat_indices = flat_indices[np.argsort(np.fft.ifftshift(F).ravel()[flat_indices])[::-1]]

# Convert to N-dimensional indices
nd_indices = np.unravel_index(flat_indices, F.shape)

print(flat_indices)   # [5 7 4] (flat)
print(np.array(nd_indices).T)     # (array([1, 2, 1]), array([2, 1, 1]))  → (1,2), (2,1), (1,1)
print(np.fft.ifftshift(F)[nd_indices])  # [9 6 5]

In [ ]:
import numpy as np

def top_n_coefficients(F, n):
    # Get amplitudes (absolute values of complex coefficients)
    amplitudes = np.abs(F)
    
    # Get flat indices of top N amplitudes
    flat_indices = np.argpartition(amplitudes.ravel(), -n)[-n:]
    
    # Sort by amplitude (descending)
    flat_indices = flat_indices[np.argsort(amplitudes.ravel()[flat_indices])[::-1]]
    
    # Convert flat indices to 3D indices
    indices_3d = np.unravel_index(flat_indices, F.shape)
    
    # Extract the top N coefficients and their amplitudes
    top_coeffs = F[indices_3d]
    top_amplitudes = amplitudes[indices_3d]
    
    return indices_3d, top_coeffs, top_amplitudes

# Example usage
indices, coeffs, amplitudes = top_n_coefficients(F, n=10)

fts = np.fft.fftshift(np.fft.fftfreq(n_modes[0], 1/n_modes[0]))
fys = np.fft.fftshift(np.fft.fftfreq(n_modes[1], 1/n_modes[1]))
fxs = np.fft.fftshift(np.fft.fftfreq(n_modes[2], 1/n_modes[2]))
fTT, fYY, fXX = np.meshgrid(fts, fys, fxs, indexing="ij")
for i in range(10):
    idx = tuple(indices[j][i] for j in range(3))
    print(f"Index: {idx}, Amplitude: {amplitudes[i]:.4f}, Coeff: {coeffs[i]:.4f}")
    print(f"  Frequencies: ft={fTT[idx]:.2f}, fy={fYY[idx]:.2f}, fx={fXX[idx]:.2f}")

In [ ]:
# ======================================================
# Plotting: grid of 2-D histograms (Re vs Im)
# ======================================================

# ------------------------------------------------------------------
# Pre-compute ground truth Gaussian parameters for each mode
# The flux grid TT, YY, XX and fluxgt must already be defined, e.g.:
#   xx = np.linspace(0, 1, Nx, endpoint=False); yy = np.linspace(0, 1, Ny, endpoint=False); tt = np.linspace(0, 1, Nt, endpoint=False)
#   TT, YY, XX = np.meshgrid(tt, yy, xx, indexing="ij")
#   fluxgt = fluxfuncgt(XX, YY, TT)          # shape (Nt, Ny, Nx)
#
# For amplitude sweeps the flux shape is proportional to amplitude, so
# we store a *normalised* version and scale per amplitude level below.
# ------------------------------------------------------------------

dvol = (1 / Nt) * (1 / Ny) * (1 / Nx)  # volume element of the grid; needed for normalisation of sums to integrals

gt_params = []   # list over modes: (mu_re, mu_im, var_re, var_im, cov)
# for mode in PROBE_MODES:
#     ft, fy, fx = mode
#     # Phase argument  2π f·x  evaluated on the full grid
#     phase = 2 * np.pi * (ft * TT + fy * YY + fx * XX)
#     cos_phase = np.cos(phase)
#     sin_phase = np.sin(phase)

#     # GT Fourier coefficient (mean of Re and Im)
#     # E[NUFFT] = sum_x e^{-i2π f·x} λ(x) Δvol — a discrete sum
#     mu_re_norm = np.sum(cos_phase  * fluxgt) * dvol
#     mu_im_norm = np.sum(-sin_phase * fluxgt) * dvol

#     # Second-moment sums (normalised)
#     var_re_norm  = np.sum(cos_phase ** 2 * fluxgt) * dvol
#     var_im_norm  = np.sum(sin_phase ** 2 * fluxgt) * dvol
#     cov_norm     = np.sum(-cos_phase * sin_phase * fluxgt) * dvol

#     gt_params.append((mu_re_norm, mu_im_norm,
#                       var_re_norm, var_im_norm, cov_norm))

# quad integrate instead of Riemann sum above (already has numerical error when
# Gaussain ball starts moving)
from scipy import integrate
for mode in tqdm(PROBE_MODES):
    ft, fy, fx = mode

    def integrand_mu_re(t, y, x):
        phase = 2 * np.pi * (ft * t + fy * y + fx * x)
        return np.cos(phase) * fluxfuncgt(x, y, t)

    def integrand_mu_im(t, y, x):
        phase = 2 * np.pi * (ft * t + fy * y + fx * x)
        return -np.sin(phase) * fluxfuncgt(x, y, t)

    def integrand_var_re(t, y, x):
        phase = 2 * np.pi * (ft * t + fy * y + fx * x)
        return np.cos(phase)**2 * fluxfuncgt(x, y, t)

    def integrand_var_im(t, y, x):
        phase = 2 * np.pi * (ft * t + fy * y + fx * x)
        return np.sin(phase)**2 * fluxfuncgt(x, y, t)

    def integrand_cov(t, y, x):
        phase = 2 * np.pi * (ft * t + fy * y + fx * x)
        return -np.cos(phase) * np.sin(phase) * fluxfuncgt(x, y, t)

    ranges = [(0, 1), (0, 1), (0, 1)]   # t in [0,1], y in [0,1], x in [0,1]
    opts = {"limit": 50}

    mu_re,   _ = integrate.nquad(integrand_mu_re,  ranges, opts=opts)
    mu_im,   _ = integrate.nquad(integrand_mu_im,  ranges, opts=opts)
    var_re,  _ = integrate.nquad(integrand_var_re, ranges, opts=opts)
    var_im,  _ = integrate.nquad(integrand_var_im, ranges, opts=opts)
    cov,     _ = integrate.nquad(integrand_cov,    ranges, opts=opts)

    gt_params.append((mu_re, mu_im, var_re, var_im, cov))

In [ ]:
n_amps  = len(AMPLITUDE_LEVELS)
n_modes = len(PROBE_MODES)
NBINS   = 40

for ai, amp in enumerate(AMPLITUDE_LEVELS):
    for mi in range(n_modes):

        fig = plt.figure(figsize=(4, 4))
        inner_gs = gridspec.GridSpec(
            2, 2,
            figure=fig,
            width_ratios=[4, 1],
            height_ratios=[1, 4],
            hspace=0.0,
            wspace=0.0,
        )

        ax      = fig.add_subplot(inner_gs[1, 0])
        ax_top  = fig.add_subplot(inner_gs[0, 0], sharex=ax)
        ax_rgt  = fig.add_subplot(inner_gs[1, 1], sharey=ax)

        c = coeffs[ai][mi]
        re, im = c.real, c.imag

        # ---- ground truth params ----
        mu_re_n, mu_im_n, var_re_n, var_im_n, cov_n = gt_params[mi]
        mu_re, mu_im = mu_re_n * amp, mu_im_n * amp
        var_re, var_im = var_re_n * amp, var_im_n * amp
        cov = cov_n * amp

        # ---- define ranges centered on mu ----
        half_re = 4 * np.sqrt(var_re)
        half_im = 4 * np.sqrt(var_im)
        # half_re = 200
        # half_im = 200
        re_range = (mu_re - half_re, mu_re + half_re)
        im_range = (mu_im - half_im, mu_im + half_im)

        # ---- main 2D histogram ----
        ax.hist2d(re, im, bins=NBINS, cmap="gray", edgecolor='face', range=[re_range, im_range])

        print(f"Comparison for mode {PROBE_MODES[mi]}")
        print(f"\tGT mean (Re, Im) for mode {PROBE_MODES[mi]}: ({mu_re:.4f}, {mu_im:.4f})")
        print(f"\tGT var (Re, Im) and cov: var_re={var_re:.4f}, var_im={var_im:.4f}, cov={cov:.4f}")
        print(f"\tSimulated mean (Re, Im) for mode {PROBE_MODES[mi]}: ({re.mean():.4f}, {im.mean():.4f})")
        print(f"\tSimulated var (Re, Im) and cov: var_re={re.var():.4f}, var_im={im.var():.4f}, cov={np.cov(re, im)[0,1]:.4f}")
        print(f"--------------------")

        # ---- Gaussian contours and GT mean ----
        mu   = np.array([mu_re, mu_im])
        cov2 = np.array([[var_re, cov],
                         [cov,    var_im]])

        gaussian_contour_levels(mu, cov2, ax, n_sigma=(1, 2, 3, 4), color="red", lw=0.8)
        ax.scatter(mu_re, mu_im, color="red", s=5, label="GT mean")

        # ---- Re marginal histogram (top) ----
        bins_re = np.linspace(re_range[0], re_range[1], NBINS + 1)
        counts_re, _ = np.histogram(re, bins_re)
        pdf_re = norm.pdf(bins_re[:-1], mu_re, np.sqrt(var_re))
        ax_top.bar(bins_re[:-1], counts_re / (counts_re.sum() * (bins_re[1] - bins_re[0])),
                   width=np.diff(bins_re), align="edge", color="steelblue", edgecolor='steelblue')
        ax_top.plot(bins_re[:-1], pdf_re, color="red", linewidth=1.2)

        # ---- Im marginal histogram (right) ----
        bins_im = np.linspace(im_range[0], im_range[1], NBINS + 1)
        counts_im, _ = np.histogram(im, bins_im)
        pdf_im = norm.pdf(bins_im[:-1], mu_im, np.sqrt(var_im))
        ax_rgt.barh(bins_im[:-1], counts_im / (counts_im.sum() * (bins_im[1] - bins_im[0])),
                    height=np.diff(bins_im), align="edge", color="steelblue", edgecolor='steelblue')
        ax_rgt.plot(pdf_im, bins_im[:-1], color="red", linewidth=1.2)

        # ---- cosmetics ----
        plt.setp(ax_top.get_xticklabels(), visible=False)
        plt.setp(ax_rgt.get_yticklabels(), visible=False)
        ax_top.tick_params(labelsize=5, left=False, labelleft=False, bottom=False, labelbottom=False)
        ax_rgt.tick_params(labelsize=5, bottom=False, labelbottom=False, left=False, labelleft=False)

        for spine in ax_top.spines.values():
            spine.set_visible(False)
        for spine in ax_rgt.spines.values():
            spine.set_visible(False)

        ax.xaxis.set_major_locator(MaxNLocator(nbins=2, prune=None))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=2, prune=None))

        ax.tick_params(labelsize=15)

        mode_str = str(PROBE_MODES[mi])
        avg_ph   = np.mean(mean_photons[ai])
        # ax_top.set_title(f"mode {mode_str}, amp={amp:.0e}, <N>={avg_ph:.0f}", fontsize=7)
        # ax.set_xlabel("Re", fontsize=7)
        # ax.set_ylabel("Im", fontsize=7)

        # ---- center on mu ----
        # half_re = 4 * np.sqrt(var_re)
        # half_im = 4 * np.sqrt(var_im)
        # ax.set_xlim(mu_re - half_re, mu_re + half_re)
        # ax.set_ylim(mu_im - half_im, mu_im + half_im)

        fname = f"output/plots/normalitycont_ftfyfx{mode_str}_avgph{avg_ph:.0f}.pdf"
        plt.savefig(fname, dpi=150, bbox_inches="tight")
        plt.close(fig)

## discrete space normality

In [ ]:
# Fourier modes to probe (t-freq, y-freq, x-freq) — zero-based, centred convention
PROBE_MODES = [
    (0, 0, 1),
    (1, 31, 0),
    (10, 5, 5),
    (5, 3, 7),
]
# lol
AMPLITUDE_LEVELS = np.array([0.000049, 0.000489, 0.004981, 0.062975])

n_modes = len(PROBE_MODES)
poisson_rates = np.zeros((100, 32, 32))
T, H, W = poisson_rates.shape
tt, yy, xx = np.linspace(0, 1, T, endpoint=False), np.linspace(0, 1, H, endpoint=False), np.linspace(0, 1, W, endpoint=False)
TT, YY, XX = np.meshgrid(tt, yy, xx, indexing="ij")
fluxfuncgt = flux_func_gaussian_ball_linear(**GAUSS_LINEAR_BASE, max_amplitude=1.0)[0]
fluxgt = fluxfuncgt(XX, YY, TT)
meangt = np.mean(fluxgt)

N_TRIALS = 10000
coeffs = {
    ai: {mi: [] for mi in range(len(PROBE_MODES))}
    for ai in range(len(AMPLITUDE_LEVELS))
}
mean_photons = {ai: [] for ai in range(len(AMPLITUDE_LEVELS))}
for ai, amp in enumerate(AMPLITUDE_LEVELS):
    print(f"Amplitude {amp:.0e}  ({ai+1}/{len(AMPLITUDE_LEVELS)})")
    poisson_rates = fluxgt * (amp / meangt)
    mean_photons[ai] = np.sum(1 - np.exp(-poisson_rates))
    mean_trials = 0
    for i in tqdm(range(N_TRIALS)):
        # p = np.random.poisson(poisson_rates)
        p = np.random.binomial(1, 1 - np.exp(-poisson_rates))   # alternative way to sample 0/1 with same mean
        # p[p > 1] = 1
        mean_trials += np.count_nonzero(p)
        F = np.fft.rfftn(p)
        for mi, mode in enumerate(PROBE_MODES):
            ft, fy, fx = mode
            idx = mode_index(mode, F.shape)
            coeff = F[idx]
            coeffs[ai][mi].append(coeff)
    print(f"analytical mean vs empirical mean photons: {mean_photons[ai]:.1f} vs {mean_trials / N_TRIALS:.1f}")
    for mi in range(len(PROBE_MODES)):
        coeffs[ai][mi] = np.array(coeffs[ai][mi])

In [ ]:
k = 10
# Flat indices (into flattened array)
flat_indices = np.argpartition(F.ravel(), -k)[-k:]
flat_indices = flat_indices[np.argsort(F.ravel()[flat_indices])[::-1]]

# Convert to N-dimensional indices
nd_indices = np.unravel_index(flat_indices, F.shape)

print(flat_indices)   # [5 7 4] (flat)
print(np.array(nd_indices).T)     # (array([1, 2, 1]), array([2, 1, 1]))  → (1,2), (2,1), (1,1)
print(F[nd_indices])  # [9 6 5]

In [ ]:
Nt, Ny, Nx = fluxgt.shape
tt = np.arange(Nt) / Nt
yy = np.arange(Ny) / Ny
xx = np.arange(Nx) / Nx
TT, YY, XX = np.meshgrid(tt, yy, xx, indexing="ij")

gt_params = {}
for ai, amp in enumerate(AMPLITUDE_LEVELS):
    # Bernoulli probability for each cell (after dead-time clipping p > 1 → 1)
    bern_p = 1 - np.exp(-fluxgt * (amp / meangt))
    for mi, mode in enumerate(PROBE_MODES):
        ft, fy, fx = mode
        phase = 2 * np.pi * (ft * TT + fy * YY + fx * XX)
        cos_phase = np.cos(phase)
        sin_phase = np.sin(phase)

        # E[F_f] = sum_x e^{-i2pi f.x} * p(x)
        # No dvol needed: FFT is a sum, not a normalised integral
        mu_re = np.sum(cos_phase  * bern_p)
        mu_im = np.sum(-sin_phase * bern_p)

        # Var[Re] = sum_x cos^2(phase_x) * p(x)(1 - p(x))
        # Var[Im] = sum_x sin^2(phase_x) * p(x)(1 - p(x))
        # Cov     = sum_x -cos*sin(phase_x) * p(x)(1 - p(x))
        bern_var = bern_p * (1 - bern_p)   # pixel-wise Bernoulli variance
        var_re = np.sum(cos_phase**2       * bern_var)
        var_im = np.sum(sin_phase**2       * bern_var)
        cov    = np.sum(-cos_phase * sin_phase * bern_var)

        gt_params[(ai, mi)] = (mu_re, mu_im, var_re, var_im, cov)

In [ ]:
NBINS = 40

for ai in range(len(AMPLITUDE_LEVELS)):
    for mi in range(n_modes):
        fig = plt.figure(figsize=(4, 4))
        inner_gs = gridspec.GridSpec(
            2, 2,
            figure=fig,
            width_ratios=[4, 1],
            height_ratios=[1, 4],
            hspace=0.0,
            wspace=0.0,
        )

        ax      = fig.add_subplot(inner_gs[1, 0])
        ax_top  = fig.add_subplot(inner_gs[0, 0], sharex=ax)
        ax_rgt  = fig.add_subplot(inner_gs[1, 1], sharey=ax)

        c = coeffs[ai][mi]
        re, im = c.real, c.imag

        # ---- ground truth params ----
        mu_re_n, mu_im_n, var_re_n, var_im_n, cov_n = gt_params[(ai, mi)]
        mu_re, mu_im = mu_re_n, mu_im_n
        var_re, var_im = var_re_n, var_im_n
        cov = cov_n

        # ---- define ranges centered on mu ----
        half_re = 4 * np.sqrt(var_re)
        half_im = 4 * np.sqrt(var_im)
        re_range = (mu_re - half_re, mu_re + half_re)
        im_range = (mu_im - half_im, mu_im + half_im)

        # ---- main 2D histogram ----
        ax.hist2d(re, im, bins=NBINS, cmap="gray", edgecolor='face', range=[re_range, im_range])

        print(f"Comparison for mode {PROBE_MODES[mi]}")
        print(f"\tGT mean (Re, Im) for mode {PROBE_MODES[mi]}: ({mu_re:.4f}, {mu_im:.4f})")
        print(f"\tGT var (Re, Im) and cov: var_re={var_re:.4f}, var_im={var_im:.4f}, cov={cov:.4f}")
        print(f"\tSimulated mean (Re, Im) for mode {PROBE_MODES[mi]}: ({re.mean():.4f}, {im.mean():.4f})")
        print(f"\tSimulated var (Re, Im) and cov: var_re={re.var():.4f}, var_im={im.var():.4f}, cov={np.cov(re, im)[0,1]:.4f}")
        print(f"--------------------")

        # ---- ground truth plots ----
        mu   = np.array([mu_re, mu_im])
        cov2 = np.array([[var_re, cov],
                            [cov,    var_im]])

        gaussian_contour_levels(mu, cov2, ax, n_sigma=(1, 2, 3, 4), color="red", lw=0.8)
        ax.scatter(mu_re, mu_im, color="red", s=5, label="GT mean")

        # ---- Re marginal histogram (top) ----
        bins_re = np.linspace(*re_range, NBINS + 1)
        counts_re, _ = np.histogram(re, bins_re)
        pdf_re = norm.pdf(bins_re[:-1], mu_re, np.sqrt(var_re))
        ax_top.bar(bins_re[:-1], counts_re / (counts_re.sum() * (bins_re[1] - bins_re[0])),
                    width=np.diff(bins_re), align="edge", color="steelblue", edgecolor='steelblue')
        ax_top.plot(bins_re[:-1], pdf_re, color="red", linewidth=1.2)

        # ---- Im marginal histogram (right) ----
        bins_im = np.linspace(*im_range, NBINS + 1)
        counts_im, _ = np.histogram(im, bins_im)
        pdf_im = norm.pdf(bins_im[:-1], mu_im, np.sqrt(var_im))
        ax_rgt.barh(bins_im[:-1], counts_im / (counts_im.sum() * (bins_im[1] - bins_im[0])),
                    height=np.diff(bins_im), align="edge", color="steelblue", edgecolor='steelblue')
        ax_rgt.plot(pdf_im, bins_im[:-1], color="red", linewidth=1.2)

        # ---- cosmetics ----
        plt.setp(ax_top.get_xticklabels(), visible=False)
        plt.setp(ax_rgt.get_yticklabels(), visible=False)
        ax_top.tick_params(labelsize=5, left=False, labelleft=False, bottom=False, labelbottom=False)
        ax_rgt.tick_params(labelsize=5, bottom=False, labelbottom=False, left=False, labelleft=False)

        for spine in ax_top.spines.values():
            spine.set_visible(False)
        for spine in ax_rgt.spines.values():
            spine.set_visible(False)

        ax.tick_params(labelsize=10)

        mode_str = str(PROBE_MODES[mi])

        # ---- center on mu ----
        ax.set_xlim(mu_re - half_re, mu_re + half_re)
        ax.set_ylim(mu_im - half_im, mu_im + half_im)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=2, prune=None))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=2, prune=None))

        fname = f"output/plots/normalitydisc_ftfyfx{mode_str}_avgph{round(mean_photons[ai])}_{T}x{H}x{W}.pdf"
        plt.savefig(fname, dpi=150, bbox_inches="tight")
        plt.close(fig)

## CFAR observed vs true bound difference

In [ ]:
from scipy import integrate
def compute_gt_fourier(fluxfunc, n_modes, lam_scale=1.0):
    """
    Compute ground truth Fourier coefficients matching finufft.nufft3d1 convention.
    
    F[k_t, k_y, k_x] = integral over [0,1]^3 of lam(x,y,t) * exp(-i*2pi*(k_t*t + k_y*y + k_x*x)) dx dy dt

    Args:
        fluxfunc: callable(x, y, t) -> float, the UNscaled flux function
        n_modes: tuple (Nt, Ny, Nx) matching finufft n_modes argument
        lam_scale: scalar, same scale factor applied to fluxfunc to get lam

    Returns:
        F_gt: complex array of shape n_modes, matching finufft output layout
    """
    Nt, Ny, Nx = n_modes

    # Mode indices: finufft uses [-N//2, ..., N//2-1] for even N
    kt_vals = np.fft.fftshift(np.fft.fftfreq(Nt, d=1.0/Nt))
    ky_vals = np.fft.fftshift(np.fft.fftfreq(Ny, d=1.0/Ny))
    kx_vals = np.fft.fftshift(np.fft.fftfreq(Nx, d=1.0/Nx))

    F_gt = np.zeros((Nt, Ny, Nx), dtype=complex)

    pbar = tqdm(total=Nt * Ny * Nx, desc="Computing GT Fourier coefficients")
    for i, kt in enumerate(kt_vals):
        for j, ky in enumerate(ky_vals):
            for k, kx in enumerate(kx_vals):

                def integrand_real(t, y, x):
                    phase = 2 * np.pi * (kt * t + ky * y + kx * x)
                    return lam_scale * fluxfunc(x, y, t) * np.cos(phase)

                def integrand_imag(t, y, x):
                    phase = 2 * np.pi * (kt * t + ky * y + kx * x)
                    return lam_scale * fluxfunc(x, y, t) * np.sin(phase)

                real_part, _ = integrate.tplquad(
                    integrand_real,
                    0, 1,        # x limits
                    0, 1,        # y limits
                    0, 1,        # t limits
                    epsabs=1e-6, epsrel=1e-6,
                )
                imag_part, _ = integrate.tplquad(
                    integrand_imag,
                    0, 1,
                    0, 1,
                    0, 1,
                    epsabs=1e-6, epsrel=1e-6,
                )

                F_gt[i, j, k] = real_part - 1j * imag_part
                pbar.update(1)
    return F_gt

def compute_gt_fourier_fast(fluxfunc, n_modes, lam_scale=1.0, grid_res=200):
    """
    Fast version using a dense quadrature grid + FFT.
    Evaluates fluxfunc on a grid_res^3 grid, then takes an FFT.
    
    Equivalent to tplquad version when grid_res is large enough relative
    to the bandwidth of fluxfunc.
    """
    Nt, Ny, Nx = n_modes

    # Use at least n_modes resolution, more for accuracy
    Gt = max(Nt, grid_res)
    Gy = max(Ny, grid_res)
    Gx = max(Nx, grid_res)

    # Cell-centered points on [0, 1)
    t = (np.arange(Gt) + 0.5) / Gt
    y = (np.arange(Gy) + 0.5) / Gy
    x = (np.arange(Gx) + 0.5) / Gx

    T, Y, X = np.meshgrid(t, y, x, indexing='ij')  # shape (Gt, Gy, Gx)
    
    # Evaluate scaled flux on grid
    lam_grid = lam_scale * fluxfunc(X, Y, T)        # shape (Gt, Gy, Gx)

    # DFT with isign=-1: F[k] = sum_j f[j] * exp(-2pi*i*k*j/N) / N^3
    # np.fft.fftn gives sum without 1/N, so divide by total points
    F_full = np.fft.fftn(lam_grid) / (Gt * Gy * Gx)

    # fftn has mode order [0, 1, ..., N-1], need to fftshift to [-N/2, ..., N/2-1]
    # then crop to the requested n_modes
    F_shifted = np.fft.fftshift(F_full)  # now centered at DC

    # Crop to n_modes (centered crop)
    st = (Gt - Nt) // 2
    sy = (Gy - Ny) // 2
    sx = (Gx - Nx) // 2
    F_gt = F_shifted[st:st+Nt, sy:sy+Ny, sx:sx+Nx].copy()

    return F_gt

In [ ]:
# ======================================================
# Configuration
# ======================================================

# Amplitude levels to sweep (scales max_amplitude)
# hardcode attempt to keep E[nphotons] = 5, 50, 500
AMPLITUDE_LEVELS = (5 / 5.038) * (5 / 6.2) * np.geomspace(1e4, 4e6, 100)

N_TRIALS = 1000          # number of Monte-Carlo trials per amplitude level

# Base Gaussian-ball args (max_amplitude will be overridden in the loop)
GAUSS_LINEAR_BASE = dict(
    start_pos=(0.2, 0.2),
    end_pos=(0.8, 0.8),
    T_exp=T_exp,
    sigma=0.1,
    normalize=False
)

NUFFT_EPS = 1e-6

# Reconstruction grid sizes
n_modes = (100, 32, 32)  # we are not probing up to the resolution Nyquist

# fluxgt = ihpp_data.flux_func_gaussian_ball(**GAUSS_BASE, max_amplitude=1.0)[0](XX, YY, TT)
fluxfuncgt = flux_func_gaussian_ball_linear(**GAUSS_LINEAR_BASE, max_amplitude=1.0)[0]
# fluxgt = fluxfuncgt(XX, YY, TT)
flux_integ_norm, _ = integrate.nquad(fluxfuncgt, [(0, 1), (0, 1), (0, 1)], opts={"limit": 50})
F_gt_norm = compute_gt_fourier_fast(fluxfuncgt, n_modes, grid_res=400)


# ======================================================
# Main simulation loop
# ======================================================

alpha = 1e-4
sigma = 3

mean_photons = []
mean_photons_obs = []
frac_F_jacc = []
frac_F_jacc_overest = []
frac_F_jacc_underest = []


for ai, amp in tqdm(list(enumerate(AMPLITUDE_LEVELS))):
    print(f"Amplitude {amp:.0e}  ({ai+1}/{len(AMPLITUDE_LEVELS)})")

    # lam, lmax = ihpp_data.flux_func_gaussian_ball(
    #     max_amplitude=amp, **GAUSS_BASE
    # )
    lam, lmax = flux_func_gaussian_ball_linear(
        max_amplitude=amp, **GAUSS_LINEAR_BASE
    )
    flux_integ = flux_integ_norm * amp
    F_gt = F_gt_norm * amp
    print(f"  theoretical mean photons = {flux_integ:.1f}")
    mean_photons.append(flux_integ)
    avg_obs = 0

    frac_F_jacc_amp = []
    frac_F_jacc_overest_amp = []
    frac_F_jacc_underest_amp = []
    for trial in range(N_TRIALS):
        points = ihpp_sim.simulate_ihpp3d(
            lambda_fn=lam,
            s_region=[[0, 0], [1, 0], [1, 1], [0, 1]],
            t_range=(0, 1),
            npoints=None,
            nx=100, ny=100, nt=100,
            lmax=lmax,
            rng=None,          # random seed each trial
        )

        n_photons = len(points)
        avg_obs += n_photons
        if n_photons == 0:
            continue

        px, py, pt = points[:, 0], points[:, 1], points[:, 2]

        F = finufft.nufft3d1(
            pt * 2 * np.pi,
            py * 2 * np.pi,
            px * 2 * np.pi,
            np.ones(n_photons, dtype="complex128"),
            n_modes=n_modes,
            eps=NUFFT_EPS,
            isign=-1,
        )
        observed_std = n_photons ** 0.5

        bd_true = flux_probing.compute_amplitude_bound(flux_integ, V=1.0, alpha=alpha)
        bd_obs_overest = flux_probing.compute_amplitude_bound(n_photons + sigma * observed_std, V=1.0, alpha=alpha)
        bd_obs_underest = flux_probing.compute_amplitude_bound(max(0, n_photons - sigma * observed_std), V=1.0, alpha=alpha)
        bd_obs = flux_probing.compute_amplitude_bound(n_photons, V=1.0, alpha=alpha)
        n_Fbd_true_mask = np.abs(F) > bd_true
        n_Fbd_obs_mask = np.abs(F) > bd_obs
        n_Fbd_obs_overest_mask = np.abs(F) > bd_obs_overest
        n_Fbd_obs_underest_mask = np.abs(F) > bd_obs_underest
        # jaccard index
        intersection = (n_Fbd_true_mask & n_Fbd_obs_mask).sum()
        union = (n_Fbd_true_mask | n_Fbd_obs_mask).sum()
        if union > 0:
            frac_F_jacc_amp.append(intersection / union)
        intersection_overest = (n_Fbd_true_mask & n_Fbd_obs_overest_mask).sum()
        union_overest = (n_Fbd_true_mask | n_Fbd_obs_overest_mask).sum()
        if union_overest > 0:
            frac_F_jacc_overest_amp.append(intersection_overest / union_overest)
        intersection_underest = (n_Fbd_true_mask & n_Fbd_obs_underest_mask).sum()
        union_underest = (n_Fbd_true_mask | n_Fbd_obs_underest_mask).sum()
        if union_underest > 0:
            frac_F_jacc_underest_amp.append(intersection_underest / union_underest)

    avg_obs = avg_obs / N_TRIALS
    mean_photons_obs.append(avg_obs)
    frac_F_jacc.append(np.array(frac_F_jacc_amp))
    frac_F_jacc_overest.append(np.array(frac_F_jacc_overest_amp))
    frac_F_jacc_underest.append(np.array(frac_F_jacc_underest_amp))
    print(f"  mean photons = {avg_obs:.1f}")
    print(f"  theoretical mean photons = {flux_integ:.1f}")

frac_F_jacc = np.array(frac_F_jacc)
frac_F_jacc_overest = np.array(frac_F_jacc_overest)
frac_F_jacc_underest = np.array(frac_F_jacc_underest)

In [ ]:
mean_photons = np.array(mean_photons)
mean_photons_obs = np.array(mean_photons_obs)
np.savez("output/plots/normality_sweep_results.npz",
    amps=AMPLITUDE_LEVELS,
    mean_photons=mean_photons,
    mean_photons_obs=mean_photons_obs,
    frac_F_jacc=frac_F_jacc,
    frac_F_jacc_overest=frac_F_jacc_overest,
    frac_F_jacc_underest=frac_F_jacc_underest,
    sigma=sigma
)

In [ ]:
import matplotlib
from matplotlib.patches import Patch

fig, ax = plt.subplots()

scale = 100 * 100 * 10000
x = np.array(mean_photons) / scale

# Compute means and stds
mean_jacc       = np.mean(frac_F_jacc, axis=1)
std_jacc        = np.std(frac_F_jacc, axis=1)

mean_over       = np.mean(frac_F_jacc_overest, axis=1)
std_over        = np.std(frac_F_jacc_overest, axis=1)

mean_under      = np.mean(frac_F_jacc_underest, axis=1)
std_under       = np.std(frac_F_jacc_underest, axis=1)

# Lines
line_black  = ax.plot(x, mean_jacc,  color="black",  label=r"Bound with $|\mathcal{P}|$")[0]
line_red    = ax.plot(x, mean_over,  color="red",    label=r"Bound with $|\mathcal{P}|+3\sqrt{|\mathcal{P}|}$")[0]
line_blue = ax.plot(x, mean_under, color="blue", label=r"Bound with $|\mathcal{P}|-3\sqrt{|\mathcal{P}|}$")[0]

ci_black  = Patch(color="black",  alpha=0.15)
ci_red    = Patch(color="red",    alpha=0.15)
ci_blue   = Patch(color="blue",   alpha=0.15)

# Variance bands (±1 std)
ax.fill_between(x, mean_jacc  - std_jacc,  np.clip(mean_jacc + std_jacc, 0, 1),  color="black",  alpha=0.15)
ax.fill_between(x, mean_over  - std_over,  mean_over  + std_over,  color="red",    alpha=0.15)
ax.fill_between(x, mean_under - std_under, mean_under + std_under, color="blue", alpha=0.15)

ax.set_xscale('log')
ax.set_xlabel(r'ppp', fontsize=10.5)
ax.set_ylabel(r'IoU vs oracle bound with $\langle 1,\phi\rangle$ ($\uparrow$)', fontsize=10.5)

# Y-axis: starting values of each line + 1
y_ticks = sorted({mean_jacc[0], mean_over[0], mean_under[0], 1.0})
ax.set_yticks(y_ticks)
ax.yaxis.set_major_formatter(plt.FormatStrFormatter('%.2f'))

ax.legend(
    handles=[line_black, ci_black, line_red, ci_red, line_blue, ci_blue],
    labels=[
        r"$|\mathcal{P}|$",         r"$\pm1\sigma$",
        r"$|\mathcal{P}|+3\sqrt{|\mathcal{P}|}$", r"$\pm1\sigma$",
        r"$|\mathcal{P}|-3\sqrt{|\mathcal{P}|}$", r"$\pm1\sigma$",
    ],
)

secax = ax.secondary_xaxis('top', functions=(lambda x: x * scale, lambda x: x / scale))
secax.set_xlabel(r'Total photons $\langle 1,\phi\rangle$', fontsize=10.5)
fig.savefig("output/plots/normality_sweep_iou.pdf", dpi=300, bbox_inches="tight")

In [ ]:
A = np.load("output/plots/normality_sweep_results.npz")
mean_photons = A["mean_photons"]
mean_photons_obs = A["mean_photons_obs"]
frac_F_jacc = A["frac_F_jacc"]
frac_F_jacc_overest = A["frac_F_jacc_overest"]
frac_F_jacc_underest = A["frac_F_jacc_underest"]

In [ ]:
import matplotlib
from matplotlib.patches import Patch

label_fontsize = 12
tick_fontsize = 15
linewidth = 3

fig, ax = plt.subplots()

scale = 100 * 100 * 10000
x = np.array(mean_photons) / scale

# Compute medians and 10th-90th percentiles
med_jacc   = np.median(frac_F_jacc,       axis=1)
mean_jacc  = np.mean(frac_F_jacc,         axis=1)
p10_jacc   = np.percentile(frac_F_jacc,   10, axis=1)
p90_jacc   = np.percentile(frac_F_jacc,   90, axis=1)

med_over   = np.median(frac_F_jacc_overest,  axis=1)
mean_over  = np.mean(frac_F_jacc_overest,    axis=1)
p10_over   = np.percentile(frac_F_jacc_overest,  10, axis=1)
p90_over   = np.percentile(frac_F_jacc_overest,  90, axis=1)

med_under  = np.median(frac_F_jacc_underest, axis=1)
mean_under = np.mean(frac_F_jacc_underest,   axis=1)
p10_under  = np.percentile(frac_F_jacc_underest, 10, axis=1)
p90_under  = np.percentile(frac_F_jacc_underest, 90, axis=1)

# Lines
line_black = ax.plot(x, mean_jacc,  color="black", linewidth=linewidth, label=r"$|\mathcal{P}|$")[0]
line_red   = ax.plot(x, mean_over,  color="red",   linewidth=linewidth, label=r"$|\mathcal{P}|+3\sqrt{|\mathcal{P}|}$")[0]
line_blue  = ax.plot(x, mean_under, color="blue",  linewidth=linewidth, label=r"$|\mathcal{P}|-3\sqrt{|\mathcal{P}|}$")[0]

ax.axvline(0.001, color="salmon", linestyle="dashed", linewidth=linewidth)

ax.set_xscale('log')
# ax.set_xlabel(r'ppp', fontsize=label_fontsize)
# ax.set_ylabel(r'IoU vs oracle bound with $\langle 1,\phi\rangle$ ($\uparrow$)', fontsize=label_fontsize)

# Y-axis: starting values of each line + 1
y_ticks = sorted({mean_jacc[0], mean_over[0], mean_under[0], 1.0})
ax.set_yticks(y_ticks)
ax.yaxis.set_major_formatter(plt.FormatStrFormatter('%.2f'))

ax.legend(
    fontsize=tick_fontsize,
)

secax = ax.secondary_xaxis('top', functions=(lambda x: x * scale, lambda x: x / scale))
ax.tick_params(axis='both', which="major", labelsize=tick_fontsize, width=2, length=10)
ax.tick_params(axis='both', which="minor", labelsize=tick_fontsize, width=2, length=5)
secax.tick_params(axis='both', which="major", labelsize=tick_fontsize, width=2, length=10)
secax.tick_params(axis='both', which="minor", labelsize=tick_fontsize, width=2, length=5)
# secax.set_xlabel(r'total photons ($\langle 1,\phi\rangle$)', fontsize=label_fontsize)
for side in ['top', 'bottom', 'left', 'right']:
    ax.spines[side].set_linewidth(2.0)
fig.set_size_inches(7, 4)
fig.savefig("output/plots/normality_sweep_iou.pdf", dpi=300, bbox_inches="tight")

In [ ]:
50 / (100 * 100 * 10000)

In [ ]:
plt.hist(frac_F_jacc_overest[0])

In [ ]:
import seaborn as sns
import pandas as pd

# Melt the 2D array into long form: one row per (x, trial) pair
n_x, n_trials = frac_F_jacc.shape  # (len(mean_photons), n_trials)

df = pd.DataFrame(frac_F_jacc, index=x).reset_index()
df.columns = ["x"] + [f"trial_{i}" for i in range(n_trials)]
df = df.melt(id_vars="x", var_name="trial", value_name="iou")

fig, ax = plt.subplots()
sns.lineplot(data=df, x="x", y="iou", ax=ax)

In [ ]:
df

# simulation supplementary plots

## uwb 1d vs 3d comparison

In [ ]:
########## 3D FFT variable setup ##########
# exposure time (how much data to read)
T_exp = 1.0  # seconds

# reconstruction grid sizes
Nt = 500
Ny = 128
Nx = 128

########## End variable setup ##########
# must be normalized coordinates within [0, 1]
xx = np.linspace(0, 1, Nx)
yy = np.linspace(0, 1, Ny)
tt = np.linspace(0, 1, Nt)

#################### DATA READING ####################
########## Gaussian ball data: uncomment if needed ##########
def flux_func_gaussian_ball_linear(
    T_exp,
    start_pos=(0.0, 1.0),   # (x, y) in normalised [0,1] coords, top-left
    end_pos=(1.0, 0.0),     # (x, y) in normalised [0,1] coords, bottom-right
    sigma=0.1,
    max_amplitude=1e4,
    normalize=False,
):
    def flux(x, y, t):
        # linear interpolation of centre position
        frac = t / T_exp if not normalize else t   # t is already in [0,1] if normalize
        curr_x = start_pos[0] + (end_pos[0] - start_pos[0]) * frac
        curr_y = start_pos[1] + (end_pos[1] - start_pos[1]) * frac
        return max_amplitude * np.exp(
            -((x - curr_x) ** 2 + (y - curr_y) ** 2) / (2 * sigma ** 2)
        )

    return flux, max_amplitude
gauss_args = dict(
    height=1,
    width=1,
    x_dist=0.4,
    y_dist=0.4,
    T_exp=T_exp,
    hz_move_x=4.5,  # left->right->left motion in 1 second
    hz_move_y=4.5,  # vertical periods in 1 second
    hz_light=0,  # flashing frequency
    sigma=0.1,
    max_amplitude=1e6,
    normalize=False,  # normalizes all coordinates to [0, 1], and scales the flux by the volume
)
# lam, lmax = ihpp_data.flux_func_gaussian_ball(**gauss_args)
lam, lmax = flux_func_gaussian_ball_linear(
    start_pos=(0.2, 0.2), end_pos=(0.8, 0.8), T_exp=T_exp, sigma=0.1, max_amplitude=1e5
)

points = ihpp_sim.simulate_ihpp3d(
    lambda_fn=lam,
    s_region=[[0, 0], [1, 0], [1, 1], [0, 1]],   # unit square
    t_range=(0, 1),                   # time interval
    npoints=None,                         # let the process draw N ~ Poisson(∫λ)
    nx=100, ny=100, nt=100,                  # grid for integral/lmax estimate
    lmax=lmax,                          # known upper bound; skips estimating it
    rng=42
)
_px = points[:, 0]
_py = points[:, 1]
_pt = points[:, 2]

px_bin = np.floor(_px * (Nx - 1)).astype(np.int64)
py_bin = np.floor(_py * (Ny - 1)).astype(np.int64)
np.clip(px_bin, 0, Nx - 1, out=px_bin)
np.clip(py_bin, 0, Ny - 1, out=py_bin)

# Quantized coordinates on the existing grids
px_q = xx[px_bin]
py_q = yy[py_bin]

# ts_in_bin[i][j] -> timestamps for y-bin i, x-bin j
ts_in_bin = [[[] for _ in range(Nx)] for _ in range(Ny)]
for xb, yb, tb in zip(px_bin, py_bin, _pt):
    ts_in_bin[yb][xb].append(tb)

ts_in_bin = [[np.asarray(ts_in_bin[i][j], dtype=_pt.dtype) for j in range(Nx)] for i in range(Ny)]

# Quantize in all 3 dimensions
pt_bin = np.floor(_pt * (Nt - 1)).astype(np.int64)
np.clip(pt_bin, 0, Nt - 1, out=pt_bin)

# Create 3D boolean array: True if at least one event falls in this (t, y, x) bin
occupancy_3d = np.zeros((Nt, Ny, Nx), dtype=np.uint8)
occupancy_3d[pt_bin, py_bin, px_bin] = 1

# px, py, pt = points[:, 0], points[:, 1], points[:, 2]
px, py, pt = px_q, py_q, _pt
logger.info(f"Simulated {points.shape[0]} events from Gaussian ball flux.")
TT, YY, XX = np.meshgrid(tt, yy, xx, indexing="ij")
fluxgt = lam(XX, YY, TT)

F = finufft.nufft3d1(
    pt * 2 * np.pi, py * 2 * np.pi, px * 2 * np.pi,  # input coordinates in radians
    np.ones_like(px, dtype="complex128"),  # input values (all ones for counting)
    n_modes=(Nt, Ny, Nx),  # output grid size
    eps=1e-6,  # desired accuracy
    isign=-1
)

alpha = 1 / (np.prod((Nt, Ny, Nx)) / 2)
bd = flux_probing.compute_amplitude_bound(len(px), V=1, alpha=alpha)
F_bd = F.copy()
F_bd[np.abs(F) < bd] = 0
uwb3d_timeF = np.fft.ifftn(np.fft.ifftshift(F_bd), axes=(1, 2), norm="forward")

In [ ]:
F.shape

In [ ]:
flux = np.fft.ifftn(np.fft.ifftshift(F_bd), norm="forward").real

In [ ]:
fluxgt.shape

In [ ]:
x = 64
y = 64

for i in [0, 125, 250, 375, 498]:
    print(f"flux[{i}, {y}, {x}] = {flux[i, y, x]:.4f}")

    plt.imshow(fluxgt[i], vmax=fluxgt.max(), cmap="gray")
    plt.scatter(x, y, color="red")
    ax = plt.gca()
    ax.set_xlabel("")
    ax.set_ylabel("")
    # for spine in ax.spines.values():
    #     spine.set_visible(False)
    ax.set_xticks([])
    ax.set_yticks([])
    plt.savefig(f"output/synthetic/compare/fluxgt_{i/Nt}sec.pdf", bbox_inches="tight")
    plt.show()

    plt.imshow(occupancy_3d[i], cmap="gray")
    ax = plt.gca()
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticks([])
    ax.set_yticks([])
    plt.savefig(f"output/synthetic/compare/binary_{i/Nt}sec.pdf", bbox_inches="tight")
    plt.show()

In [ ]:
# gt flux with events
plt.plot(tt, fluxgt[:, y, x] / fluxgt[:, y, x].max())
plt.scatter(ts_in_bin[y][x], np.zeros_like(ts_in_bin[y][x])-0.1, color="red")
plt.axhline(-0.1, color="gray", linestyle="--")
ax = plt.gca()
plt.xlim(0, 1)
ax.set_xlabel("")
ax.set_ylabel("")
# for spine in ax.spines.values():
#     spine.set_visible(False)
ax.set_xticks([])
ax.set_yticks([])
plt.savefig("output/synthetic/compare/flux_gt.pdf", bbox_inches="tight")

In [ ]:
ft = np.fft.fftshift(np.fft.fftfreq(Nt, d=1/Nt))

In [ ]:
# gt fourier
Ft_gt = np.fft.fftshift(np.fft.fft(fluxgt[:, y, x]))
plt.plot(ft, np.abs(Ft_gt))
# plt.yscale("log")
plt.xlim(-50, 50)
ax = plt.gca()
ax.set_xlabel("")
ax.set_ylabel("")
# for spine in ax.spines.values():
#     spine.set_visible(False)
ax.set_xticks([])
ax.set_yticks([])
plt.savefig("output/synthetic/compare/Ft_gt.pdf", bbox_inches="tight")

In [ ]:
# uwb3d recon flux
plt.plot(tt, flux[:, y, x] / flux[:, y, x].max())
ax = plt.gca()
ax.set_xlabel("")
ax.set_ylabel("")
# for spine in ax.spines.values():
#     spine.set_visible(False)
ax.set_xticks([])
ax.set_yticks([])
plt.savefig("output/synthetic/compare/flux_uwb3d.pdf", bbox_inches="tight")

In [ ]:
# uwb3d fourier
ft = np.fft.fftshift(np.fft.fftfreq(Nt, d=1/Nt))
uwb3d_max = np.abs(uwb3d_timeF[:, y, x]).max()
gt_max = np.abs(Ft_gt).max()
plt.plot(ft, np.abs(np.fft.fftshift(uwb3d_timeF[:, y, x]))/uwb3d_max)
# plt.plot(ft, np.abs(Ft_gt)/gt_max)
# plt.yscale("log")
plt.xlim(-50, 50)
ax = plt.gca()
ax.set_xlabel("")
ax.set_ylabel("")
# for spine in ax.spines.values():
#     spine.set_visible(False)
ax.set_xticks([])
ax.set_yticks([])
plt.savefig("output/synthetic/compare/Ft_uwb3d.pdf", bbox_inches="tight")

In [ ]:
ts_1d = ts_in_bin[y][x]
F_1d = finufft.nufft1d1(ts_1d * 2 * np.pi, np.ones(ts_1d.shape[0], dtype="complex128"), Nt, eps=1e-6, isign=-1)
bound_1d = flux_probing.compute_amplitude_bound(ts_1d.size, 1, alpha=1/(Nt/2))
F_1d_bd = F_1d.copy()
F_1d_bd[np.abs(F_1d) < bound_1d] = 0

In [ ]:
# uwb1d fourier
plt.plot(ft, np.abs(F_1d))
plt.axhline(bound_1d, color="red", linestyle="--", label="1D UWB bound")
# plt.yscale("log")
plt.xlim(-50, 50)
ax = plt.gca()
ax.set_xlabel("")
ax.set_ylabel("")
# for spine in ax.spines.values():
#     spine.set_visible(False)
ax.set_xticks([])
ax.set_yticks([])
plt.savefig("output/synthetic/compare/Ft_uwb1d.pdf", bbox_inches="tight")

In [ ]:
# uwb1d recon flux
flux_1duwb = np.fft.ifft(np.fft.ifftshift(F_1d_bd), norm="forward").real
flux_1duwb = np.clip(flux_1duwb, 0, None)  # clip negative values to zero
plt.plot(tt, flux_1duwb / flux_1duwb.max())
ax = plt.gca()
ax.set_xlabel("")
ax.set_ylabel("")
# for spine in ax.spines.values():
#     spine.set_visible(False)
ax.set_xticks([])
ax.set_yticks([])
plt.savefig("output/synthetic/compare/flux_uwb1d.pdf", bbox_inches="tight")

# Plotting things

## FFT probing setup

In [ ]:
########## 3D FFT variable setup ##########
# exposure time (how much data to read)
T_exp = 1.0  # seconds

# reconstruction grid sizes
Nt = 1000
Ny = 256
Nx = 256

########## End variable setup ##########
# must be normalized coordinates within [0, 1]
xx = np.linspace(0, 1, Nx)
yy = np.linspace(0, 1, Ny)
tt = np.linspace(0, 1, Nt)

#################### DATA READING ####################
########## Gaussian ball data: uncomment if needed ##########
V = Nx * Ny * T_exp
gauss_args = dict(
    height=1,
    width=1,
    x_dist=0.4,
    y_dist=0.4,
    T_exp=T_exp,
    hz_move_x=2,  # left->right->left motion in 1 second
    hz_move_y=2,  # vertical periods in 1 second
    hz_light=0,  # flashing frequency
    sigma=0.1,
    max_amplitude=1e5/2,
    normalize=True,  # normalizes all coordinates to [0, 1], and scales the flux by the volume
)
lam, lmax = ihpp_data.flux_func_gaussian_ball(**gauss_args)
points = ihpp_sim.simulate_ihpp3d(
    lambda_fn=lam,
    s_region=[[0, 0], [1, 0], [1, 1], [0, 1]],   # unit square
    t_range=(0, 1),                   # time interval
    npoints=None,                         # let the process draw N ~ Poisson(∫λ)
    nx=100, ny=100, nt=100,                  # grid for integral/lmax estimate
    lmax=lmax,                          # known upper bound; skips estimating it
    rng=42
)
px, py, pt = points[:, 0], points[:, 1], points[:, 2]
logger.info(f"Simulated {points.shape[0]} events from Gaussian ball flux.")
x, y, t = points[:,0], points[:,1], points[:,2]
TT, YY, XX = np.meshgrid(tt, yy, xx, indexing="ij")
fluxgt = lam(XX, YY, TT)

In [ ]:
import cv2
frac = 0.5
cv2.imwrite(str(config.OUTPUT_DIR / f"methodfig/fluxgt{frac}.png"), (255 * fluxgt[int(frac * Nt)] / fluxgt.max()).astype("uint8"))

In [ ]:
stuff = {
    "fluxgt": fluxgt,
}
# save into npz
np.savez(config.OUTPUT_DIR / "methodfig/gaussianball-gt.npz", **stuff)

In [ ]:
grid = pv.ImageData()
grid.dimensions = np.array(fluxgt.T.shape) + 1  # VTK expects points, not cells
grid.spacing = (1.0, 1.0, 1.0)              # voxel size
grid.origin = (0.0, 0.0, 0.0)

# Flatten in Fortran order for VTK
grid.cell_data["values"] = (fluxgt / np.max(fluxgt)).T.flatten(order="F")

# Save
grid.save(config.OUTPUT_DIR / "methodfig/fluxgt.vti")

In [ ]:
cloud = pv.PolyData(points)

# optional: attach per-point data (scalars or vectors)
# cloud.point_data["values"] = values  # shape (N,)
# cloud.point_data["rgb"] = rgb      # shape (N,3)
# cloud.point_data["normal"] = normals  # shape (N,3)

cloud.save(config.OUTPUT_DIR / "methodfig/pointcloud.vtp")

In [ ]:
startpts = points[pt < 0.05]
plt.scatter(startpts[:,0], startpts[:,1], s=5)
plt.xlim(0, 1)
plt.ylim(0, 1)
# equal aspect ratio
plt.gca().set_aspect('equal', adjustable='box')
# origin upper
plt.gca().invert_yaxis()

In [ ]:
plt.imshow(fluxgt[0])

In [ ]:
fluximgoffset1 = 0.2
fluximgoffset2 = 0.5
fluximg1 = fluxgt[int(Nt * fluximgoffset1)]
fluximg2 = fluxgt[int(Nt * fluximgoffset2)]
import cv2
cv2.imwrite(f"output/fluximg1_{fluximgoffset1}.png", (fluximg1 / fluximg1.max() * 255).astype(np.uint8))
cv2.imwrite(f"output/fluximg2_{fluximgoffset2}.png", (fluximg2 / fluximg2.max() * 255).astype(np.uint8))
plt.imshow(fluximg1, cmap='gray')
plt.show()
plt.imshow(fluximg2, cmap='gray')
plt.show()

In [ ]:
from plotly import graph_objects as go
fig = go.Figure(data=go.Volume(
    x=XX.flatten(),
    y=YY.flatten(),
    z=TT.flatten(),
    value=fluxgt.flatten(),
    opacityscale=[[0, 0.5], [1, 1.0]],
    surface_count=10,
    colorscale='gray'
))
img_array = fluxgt[0]

# If image is RGB, convert to grayscale or use one channel
if len(img_array.shape) == 3:
    img_array = np.mean(img_array, axis=2)

# Create coordinate grids
height, width = img_array.shape
x = np.linspace(0, 1, width)
y = np.linspace(0, 1, height)
X, Y = np.meshgrid(x, y)

# Z is constant (surface is at z=0, aligned with xy-plane)
Z = np.zeros_like(X)

fluximgoffset1 = 0.2
fluximgoffset2 = 0.5
fluximg1 = fluxgt[int(Nt * fluximgoffset1)]

# Create surface plot
# fig.add_trace(go.Surface(
#     x=X,
#     y=Y,
#     z=Z + 0.2,
#     surfacecolor=np.ones_like(X),
#     colorscale='Blues',
#     showscale=False,
#     opacity=0.5
# ))

# # Create surface plot
# fig.add_trace(go.Surface(
#     x=X,
#     y=Y,
#     z=Z + 0.5,
#     surfacecolor=np.ones_like(X),
#     colorscale='Greens',
#     showscale=False,
#     opacity=0.5
# ))
fig.update_layout(
    scene=dict(
        xaxis=dict(showgrid=False, showbackground=False, zeroline=False, showticklabels=False, title=''),
        yaxis=dict(showgrid=False, showbackground=False, zeroline=False, showticklabels=False, title=''),
        zaxis=dict(showgrid=False, showbackground=False, zeroline=False, showticklabels=False, title=''),
        bgcolor="rgba(0,0,0,0)",
        aspectratio=dict(x=1, y=1, z=3)
    )
)
# Add black rectangular prism edge outline for [0,1]x[0,1]x[0,1]
edges = [
    [[0, 1], [0, 0], [0, 0]],  # x-axis edges
    [[0, 1], [1, 1], [0, 0]],
    [[0, 1], [0, 0], [1, 1]],
    [[0, 1], [1, 1], [1, 1]],
    [[0, 0], [0, 1], [0, 0]],  # y-axis edges
    [[1, 1], [0, 1], [0, 0]],
    [[0, 0], [0, 1], [1, 1]],
    [[1, 1], [0, 1], [1, 1]],
    [[0, 0], [0, 0], [0, 1]],  # z-axis edges
    [[1, 1], [0, 0], [0, 1]],
    [[0, 0], [1, 1], [0, 1]],
    [[1, 1], [1, 1], [0, 1]],
]

for edge in edges:
    fig.add_trace(go.Scatter3d(
        x=edge[0], y=edge[1], z=edge[2],
        mode='lines',
        line=dict(color='black', width=2),
        showlegend=False
    ))

# draw edges around [0,1]x[0,1], with z at 0.2 and 0.5

edges_02 = [
    [[0, 1], [0, 0], [0.2, 0.2]],  # x-axis edges
    [[0, 1], [1, 1], [0.2, 0.2]],
    [[0, 0], [0, 1], [0.2, 0.2]],  # y-axis edges
    [[1, 1], [0, 1], [0.2, 0.2]],
]

edges_05 = [
    [[0, 1], [0, 0], [0.5, 0.5]],  # x-axis edges
    [[0, 1], [1, 1], [0.5, 0.5]],
    [[0, 0], [0, 1], [0.5, 0.5]],  # y-axis edges
    [[1, 1], [0, 1], [0.5, 0.5]],
]

for edge in edges_02:
    fig.add_trace(go.Scatter3d(
        x=edge[0], y=edge[1], z=edge[2],
        mode='lines',
        line=dict(color='blue', width=10),
        showlegend=False
    ))

for edge in edges_05:
    fig.add_trace(go.Scatter3d(
        x=edge[0], y=edge[1], z=edge[2],
        mode='lines',
        line=dict(color='green', width=10),
        showlegend=False
    ))

fig.update_layout(title="Ground Truth Flux (Voxel Grid)")
fig.show()

## UWB with 3D NUFFT

### Run the full 3D fft

In [ ]:
F = finufft.nufft3d1(
    pt * 2 * np.pi, py * 2 * np.pi, px * 2 * np.pi,  # input coordinates in radians
    np.ones_like(px, dtype="complex128"),  # input values (all ones for counting)
    n_modes=(Nt, Ny, Nx),  # output grid size
    eps=1e-6,  # desired accuracy
    isign=-1
)

In [ ]:
alpha = 0.00001
bd = flux_probing.compute_amplitude_bound(len(px), V=1, alpha=alpha)
F_bd = F.copy()
F_bd[np.abs(F) < bd] = 0
fluxrecon = np.fft.ifftn(np.fft.ifftshift(F_bd), norm="forward").real
fluxrecon = np.clip(fluxrecon, 0, None)

In [ ]:
np.count_nonzero(~(np.abs(F) < bd))

In [ ]:
import cv2

In [ ]:
frac = 0.5
framewrite = fluxrecon[int(frac * Nt)] / fluxrecon.max() * 255
cv2.imwrite(str(config.OUTPUT_DIR / f"methodfig/fluxrecon_{int(frac * Nt)}.png"), framewrite.astype(np.uint8))

In [ ]:
grid = pv.ImageData()
grid.dimensions = np.array(fluxrecon.T.shape) + 1
grid.spacing = (1.0, 1.0, 1.0)
grid.origin = (0.0, 0.0, 0.0)
grid.cell_data["values"] = (fluxrecon / np.max(fluxrecon)).T.flatten(order="F")
grid.save(config.OUTPUT_DIR / "methodfig/fluxrecon.vti")

In [ ]:
np.c_[[1,2,3], [4,5,6]]

In [ ]:
ft = np.fft.fftshift(np.fft.fftfreq(Nt, d=1/Nt))
fy = np.fft.fftshift(np.fft.fftfreq(Ny, d=1/Ny))
fx = np.fft.fftshift(np.fft.fftfreq(Nx, d=1/Nx))
FT, FY, FX = np.meshgrid(ft, fy, fx, indexing="ij")

In [ ]:
freqinfo = {
    "F": F,
    "FT": FT,
    "FY": FY,
    "FX": FX,
}
# save as npz
np.savez(config.OUTPUT_DIR / "methodfig/gaussianball-freqinfo.npz", **freqinfo)

In [ ]:
# find the top 5 frequency triplets that have the highest magnitude
FT_flat = FT.flatten()
FY_flat = FY.flatten()
FX_flat = FX.flatten()
mag = np.abs(F.flatten())
top_indices = np.argsort(mag)[-5:][::-1]
top_freqs = np.array(list(zip(FT_flat[top_indices], FY_flat[top_indices], FX_flat[top_indices])))
print(top_freqs)
print(F.flatten()[top_indices])
bot_indices = np.argsort(mag)[:5]
bot_freqs = np.array(list(zip(FT_flat[bot_indices], FY_flat[bot_indices], FX_flat[bot_indices])))
print(bot_freqs)
print(F.flatten()[bot_indices])

In [ ]:
# check the highest abs fourier coefficient excluding DC


In [ ]:
from sptflux import config
plt.imshow(fluxrecon[0])

In [ ]:
stuff = {
    "fluxgt": fluxgt,
    "fluxrecon": fluxrecon,
    "pt": pt,
    "py": py,
    "px": px,
}
# save into npz
np.savez(config.OUTPUT_DIR / "methodfig/gaussianball-info.npz", **stuff)

In [ ]:
fluximgoffset1 = 0.2
fluximgoffset2 = 0.5
fluximg1 = fluxgt[int(Nt * fluximgoffset1)]
fluximg2 = fluxgt[int(Nt * fluximgoffset2)]

t0 = 0.48
t1 = 0.52
pts_in_window = (t >= t0) & (t <= t1)
px_window = px[pts_in_window]
py_window = py[pts_in_window]
pt_window = pt[pts_in_window]
# discretize these points into a binary 2d image
H, xedges, yedges = np.histogram2d(py_window, px_window, bins=[Ny, Nx], range=[[0, 1], [0, 1]])
binary_image = (H > 0).astype(np.uint8)
cv2.imwrite(f"output/binary_image_{t0}_{t1}.png", binary_image * 255)

In [ ]:
from plotly import graph_objects as go
fig = go.Figure(data=go.Volume(
    x=XX.flatten(),
    y=YY.flatten(),
    z=TT.flatten(),
    value=fluxrecon.flatten(),
    opacityscale=[[0, 0.1], [1, 1.0]],
    surface_count=10,
    colorscale='gray'
))
img_array = fluxgt[0]

# If image is RGB, convert to grayscale or use one channel
if len(img_array.shape) == 3:
    img_array = np.mean(img_array, axis=2)

# Create coordinate grids
height, width = img_array.shape
x = np.linspace(0, 1, width)
y = np.linspace(0, 1, height)
X, Y = np.meshgrid(x, y)

# Z is constant (surface is at z=0, aligned with xy-plane)
Z = np.zeros_like(X)

fluximgoffset1 = 0.2
fluximgoffset2 = 0.5
fluximg1 = fluxgt[int(Nt * fluximgoffset1)]

# Create surface plot
# fig.add_trace(go.Surface(
#     x=X,
#     y=Y,
#     z=Z + 0.2,
#     surfacecolor=np.ones_like(X),
#     colorscale='Blues',
#     showscale=False,
#     opacity=0.5
# ))

# # Create surface plot
# fig.add_trace(go.Surface(
#     x=X,
#     y=Y,
#     z=Z + 0.5,
#     surfacecolor=np.ones_like(X),
#     colorscale='Greens',
#     showscale=False,
#     opacity=0.5
# ))
fig.update_layout(
    scene=dict(
        xaxis=dict(showgrid=False, showbackground=False, zeroline=False, showticklabels=False, title=''),
        yaxis=dict(showgrid=False, showbackground=False, zeroline=False, showticklabels=False, title=''),
        zaxis=dict(showgrid=False, showbackground=False, zeroline=False, showticklabels=False, title=''),
        bgcolor="rgba(0,0,0,0)",
        aspectratio=dict(x=1, y=1, z=3)
    )
)
# Add black rectangular prism edge outline for [0,1]x[0,1]x[0,1]
edges = [
    [[0, 1], [0, 0], [0, 0]],  # x-axis edges
    [[0, 1], [1, 1], [0, 0]],
    [[0, 1], [0, 0], [1, 1]],
    [[0, 1], [1, 1], [1, 1]],
    [[0, 0], [0, 1], [0, 0]],  # y-axis edges
    [[1, 1], [0, 1], [0, 0]],
    [[0, 0], [0, 1], [1, 1]],
    [[1, 1], [0, 1], [1, 1]],
    [[0, 0], [0, 0], [0, 1]],  # z-axis edges
    [[1, 1], [0, 0], [0, 1]],
    [[0, 0], [1, 1], [0, 1]],
    [[1, 1], [1, 1], [0, 1]],
]

for edge in edges:
    fig.add_trace(go.Scatter3d(
        x=edge[0], y=edge[1], z=edge[2],
        mode='lines',
        line=dict(color='black', width=2),
        showlegend=False
    ))

# draw edges around [0,1]x[0,1], with z at 0.2 and 0.5

edges_02 = [
    [[0, 1], [0, 0], [0.2, 0.2]],  # x-axis edges
    [[0, 1], [1, 1], [0.2, 0.2]],
    [[0, 0], [0, 1], [0.2, 0.2]],  # y-axis edges
    [[1, 1], [0, 1], [0.2, 0.2]],
]

edges_05 = [
    [[0, 1], [0, 0], [0.5, 0.5]],  # x-axis edges
    [[0, 1], [1, 1], [0.5, 0.5]],
    [[0, 0], [0, 1], [0.5, 0.5]],  # y-axis edges
    [[1, 1], [0, 1], [0.5, 0.5]],
]

for edge in edges_02:
    fig.add_trace(go.Scatter3d(
        x=edge[0], y=edge[1], z=edge[2],
        mode='lines',
        line=dict(color='blue', width=10),
        showlegend=False
    ))

for edge in edges_05:
    fig.add_trace(go.Scatter3d(
        x=edge[0], y=edge[1], z=edge[2],
        mode='lines',
        line=dict(color='green', width=10),
        showlegend=False
    ))

fig.update_layout(title="Ground Truth Flux (Voxel Grid)")
fig.show()

In [ ]:
import plotly.graph_objects as go
def fig_spt_points(points, r=0.1, max_pts=5000) -> go.Figure:
    """
    3D scatter plot of spatio-temporal points, colored by local density (number
    of neighbors within radius r).

    Args:
        points (np.ndarray): Array of shape (N, 3) with columns (x, y, t).
        r (float): Radius to count neighbors for density estimation.
    Returns:
        fig (go.Figure): 3D scatter plot with points colored by local density.
    """
    import visu3d as v3d
    if len(points) > max_pts:
        # reduce number of points for visualization
        idx = np.random.choice(len(points), size=max_pts, replace=False)
        points = points[idx]
    cmap = plt.get_cmap("viridis")
    # # Normalize t to [0, 1] for colormap
    # t_norm = (t - t.min()) / (t.max() - t.min())
    # rgb = cm.viridis(t_norm)[:, :3]  # shape (N, 3), drop alpha
    # trace = v3d.Point3d(p=points, rgb=rgb * 255).make_traces()[0]
    trace = v3d.Point3d(p=points).make_traces()[0]
    # color the points by timestamp
    
    trace.update({"marker": {
        # "color": points[:, 2],
        "showscale": True,
        "colorscale": "inferno",
        "colorbar": {"title": "Timestamp", "len": 0.5, "y": 0.4},
    }})
    fig = v3d.make_fig([trace])
    return fig
tmp = points.copy()
# tmp[:, 2] = tmp[:, 2] * 2  # stretch time axis
fig = fig_spt_points(tmp)
# Add black rectangular prism edge outline for [0,1]x[0,1]x[0,1]
edges = [
    [[0, 1], [0, 0], [0, 0]],  # x-axis edges
    [[0, 1], [1, 1], [0, 0]],
    [[0, 1], [0, 0], [1, 1]],
    [[0, 1], [1, 1], [1, 1]],
    [[0, 0], [0, 1], [0, 0]],  # y-axis edges
    [[1, 1], [0, 1], [0, 0]],
    [[0, 0], [0, 1], [1, 1]],
    [[1, 1], [0, 1], [1, 1]],
    [[0, 0], [0, 0], [0, 1]],  # z-axis edges
    [[1, 1], [0, 0], [0, 1]],
    [[0, 0], [1, 1], [0, 1]],
    [[1, 1], [1, 1], [0, 1]],
]

for edge in edges:
    fig.add_trace(go.Scatter3d(
        x=edge[0], y=edge[1], z=edge[2],
        mode='lines',
        line=dict(color='black', width=2),
        showlegend=False
    ))
    edges_02 = [
    [[0, 1], [0, 0], [0.2, 0.2]],  # x-axis edges
    [[0, 1], [1, 1], [0.2, 0.2]],
    [[0, 0], [0, 1], [0.2, 0.2]],  # y-axis edges
    [[1, 1], [0, 1], [0.2, 0.2]],
]

edges_05 = [
    [[0, 1], [0, 0], [0.5, 0.5]],  # x-axis edges
    [[0, 1], [1, 1], [0.5, 0.5]],
    [[0, 0], [0, 1], [0.5, 0.5]],  # y-axis edges
    [[1, 1], [0, 1], [0.5, 0.5]],
]

for edge in edges_02:
    fig.add_trace(go.Scatter3d(
        x=edge[0], y=edge[1], z=edge[2],
        mode='lines',
        line=dict(color='blue', width=10),
        showlegend=False
    ))

for edge in edges_05:
    fig.add_trace(go.Scatter3d(
        x=edge[0], y=edge[1], z=edge[2],
        mode='lines',
        line=dict(color='green', width=10),
        showlegend=False
    ))
fig.update_layout(
    scene=dict(
        aspectratio=dict(x=1, y=1, z=3),
        xaxis=dict(showgrid=False, showbackground=False, zeroline=False, showticklabels=False, title=''),
        yaxis=dict(showgrid=False, showbackground=False, zeroline=False, showticklabels=False, title=''),
        zaxis=dict(showgrid=False, showbackground=False, zeroline=False, showticklabels=False, title=''),
        bgcolor="rgba(0,0,0,0)",
        aspectmode=None,
    ),
    title="Ground Truth Flux (Voxel Grid)"
)

In [ ]:
fig = v3d.make_fig()

In [ ]:
fig.to_dict()['layout']['scene']